# M1-T02: Data Integrity Check — `markdown_applied` vs `discount_pct`

**Done when (from the Task Board):**
1. Count / percentage of rows where `markdown_applied = 1` and `discount_pct = 0` is reported.
2. Interpretation is decided: *contradiction* vs. *legitimate distinct fields*.
3. Resolution is documented for the EDA Report, before Track 3B begins.

| Done-when item | Where it is answered |
|---|---|
| 1. Count / percentage | Section 2 |
| 2. Interpretation decided | Sections 3-6 (evidence) and Section 7 (verdict) |
| 3. Resolution documented | Section 8 (paste-ready text) and Section 9 (saved JSON) |


## 1. Load the data

In [1]:
import hashlib, json, sys
from pathlib import Path
import numpy as np
import pandas as pd

# Resolve project root (notebooks/ is 1 level below project root)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATASET_PATH, REPORTS_M1_DIR

DATA_PATH = RAW_DATASET_PATH
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Raw dataset not found at {DATA_PATH}."
    )

df = pd.read_csv(DATA_PATH)
print("File  :", DATA_PATH.resolve())
print("SHA256:", hashlib.sha256(DATA_PATH.read_bytes()).hexdigest())
print("Shape :", df.shape)
assert df.shape == (100_000, 42), "Unexpected shape - is this the same dataset version?"

needed = ["markdown_applied", "discount_pct", "base_price", "selling_price", "product_name"]
assert not set(needed) - set(df.columns), set(needed) - set(df.columns)
assert df[["markdown_applied", "discount_pct"]].notna().all().all(), "Unexpected nulls"
print("markdown_applied values:", sorted(df["markdown_applied"].unique().tolist()))


File  : D:\DEPI\Project\Github Repo\NHA-5-150\data\raw\perishable_goods_management.csv
SHA256: de94302b867c9debedfd45c431306623fdfc038f5ed8ca17736339b4460a6674
Shape : (100000, 42)
markdown_applied values: [0, 1]


## 2. The required number: `markdown_applied = 1` and `discount_pct = 0`

A small tolerance is used instead of `== 0`, since `discount_pct` is a float.

In [2]:
TOL = 1e-9
md_on     = df["markdown_applied"] == 1
disc_zero = df["discount_pct"].abs() <= TOL

flagged_no_discount = md_on & disc_zero

n_total  = len(df)
n_md     = int(md_on.sum())
n_bad    = int(flagged_no_discount.sum())
pct_all  = 100 * n_bad / n_total
pct_md   = 100 * n_bad / n_md

print(f"Rows total                                  : {n_total:,}")
print(f"Rows with markdown_applied = 1              : {n_md:,}")
print(f"Rows with markdown_applied = 1 & discount=0 : {n_bad:,}")
print(f"  as % of all rows                          : {pct_all:.4f} %")
print(f"  as % of markdown_applied = 1 rows         : {pct_md:.4f} %")

Rows total                                  : 100,000
Rows with markdown_applied = 1              : 30,651
Rows with markdown_applied = 1 & discount=0 : 0
  as % of all rows                          : 0.0000 %
  as % of markdown_applied = 1 rows         : 0.0000 %


## 3. Evidence A: the full 2x2 table

Checks both directions: markdown flagged without a discount, **and** a discount recorded without the flag.

In [3]:
has_disc = df["discount_pct"] > TOL
tab = pd.crosstab(df["markdown_applied"], has_disc,
                  rownames=["markdown_applied"], colnames=["discount_pct > 0"])
print(tab)

n_reverse = int((~md_on & has_disc).sum())
print("\nmarkdown_applied = 0 but discount_pct > 0:", n_reverse)

discount_pct > 0  False  True 
markdown_applied              
0                 69349      0
1                     0  30651

markdown_applied = 0 but discount_pct > 0: 0


## 4. Evidence B: what values does `discount_pct` take?

`discount_pct` is stored as a **fraction** (0.25 = 25 %), not a percent. Also looks for tiny non-zero values that could be mistaken for zero.

In [4]:
print(df.groupby("markdown_applied")["discount_pct"].describe().round(4))

disc_md = df.loc[md_on, "discount_pct"]
print(f"\nSmallest discount among markdown rows: {disc_md.min():.4f}")
print(f"Largest  discount among markdown rows: {disc_md.max():.4f}")
print(f"Distinct discount values (markdown rows): {disc_md.round(4).nunique()}")
print(f"Markdown rows with 0 < discount < 0.01  : {int(((disc_md > TOL) & (disc_md < 0.01)).sum())}")
print("\nMost common discount values:")
print(disc_md.round(2).value_counts().head(8).sort_index())

                    count    mean     std  min   25%   50%   75%   max
markdown_applied                                                      
0                 69349.0  0.0000  0.0000  0.0  0.00  0.00  0.00  0.00
1                 30651.0  0.3297  0.1723  0.1  0.19  0.25  0.46  0.75

Smallest discount among markdown rows: 0.1000
Largest  discount among markdown rows: 0.7500
Distinct discount values (markdown rows): 66
Markdown rows with 0 < discount < 0.01  : 0

Most common discount values:
discount_pct
0.16    1447
0.18    1406
0.19    1464
0.20    1473
0.21    1541
0.22    1497
0.23    1548
0.24    1430
Name: count, dtype: int64


In [5]:
bands = pd.cut(disc_md, bins=[0.0999, 0.20, 0.30, 0.40, 0.50, 0.60, 0.75],
             labels=["10-20%", "20-30%", "30-40%", "40-50%", "50-60%", "60-75%"])
depth = bands.value_counts().sort_index().to_frame("rows")
depth["share_%"] = (100 * depth["rows"] / len(disc_md)).round(1)
print(depth)

share_le30 = 100 * (disc_md <= 0.30 + TOL).mean()
n_gt50     = int((disc_md > 0.50 + TOL).sum())
share_gt50 = 100 * n_gt50 / len(disc_md)
depth_note = (f"{share_le30:.1f}% of markdown rows have a discount of 30% or less; "
              f"{n_gt50:,} rows ({share_gt50:.1f}%) are above 50%.")
print("\nFor the EDA Report:", depth_note)

              rows  share_%
discount_pct               
10-20%        9248     30.2
20-30%        9155     29.9
30-40%        2274      7.4
40-50%        3531     11.5
50-60%        3185     10.4
60-75%        3258     10.6

For the EDA Report: 60.0% of markdown rows have a discount of 30% or less; 6,443 rows (21.0%) are above 50%.


## 5. Evidence C: do the prices agree with the flag?

If `markdown_applied` and `discount_pct` were inconsistent, `selling_price` would show it.
`selling_price` is rounded to the cent, so the allowed error is half a cent relative to `base_price`.

In [6]:
price_differs = (df["base_price"] - df["selling_price"]).abs() > 1e-9
print("Share of rows where selling_price != base_price:")
print(price_differs.groupby(df["markdown_applied"]).mean().round(4).rename("share"))

implied  = 1 - df["selling_price"] / df["base_price"]
err      = (implied - df["discount_pct"]).abs()
allowed  = 0.005 / df["base_price"] + 1e-9
n_price_mismatch = int((err > allowed).sum())

print(f"\nMax |implied discount - discount_pct| : {err.max():.5f}")
print(f"Rows outside rounding tolerance        : {n_price_mismatch}")

Share of rows where selling_price != base_price:
markdown_applied
0    0.0
1    1.0
Name: share, dtype: float64

Max |implied discount - discount_pct| : 0.00490
Rows outside rounding tolerance        : 0


## 6. Evidence D: the example from the proposal

The v4 proposal mentions a Salami batch with markdown flagged but no discount. Look for it directly.

In [7]:
sal = df[df["product_name"].str.contains("salami", case=False, na=False)]
print(f"Salami rows: {len(sal):,}  |  with markdown_applied = 1: {int((sal['markdown_applied'] == 1).sum()):,}")

sal_bad = sal[(sal["markdown_applied"] == 1) & (sal["discount_pct"].abs() <= TOL)]
print(f"Salami rows with markdown_applied = 1 & discount_pct = 0: {len(sal_bad)}")

if len(sal):
    print("\ndiscount_pct for Salami rows with markdown_applied = 1:")
    print(sal.loc[sal["markdown_applied"] == 1, "discount_pct"].describe().round(3))

Salami rows: 1,626  |  with markdown_applied = 1: 465
Salami rows with markdown_applied = 1 & discount_pct = 0: 0

discount_pct for Salami rows with markdown_applied = 1:
count    465.000
mean       0.277
std        0.141
min        0.100
25%        0.180
50%        0.220
75%        0.320
max        0.750
Name: discount_pct, dtype: float64


## 7. Verdict (decided by the evidence above)

In [8]:
SMALL_SHARE = 0.001   # 0.1 % of markdown rows: threshold for "a handful of rows"

consistent = (n_bad == 0) and (n_reverse == 0) and (n_price_mismatch == 0)

if consistent:
    interpretation = "NOT A CONTRADICTION: the fields are fully consistent"
    action         = "No rows excluded or flagged. Use markdown_applied and discount_pct as they are."
elif n_reverse == 0 and n_price_mismatch == 0 and (n_bad / n_md) < SMALL_SHARE:
    interpretation = "A handful of inconsistent rows"
    action         = "Flag these rows (keep a boolean column) and exclude them from Track 3B training."
else:
    interpretation = "MEANINGFUL INCONSISTENCY: needs a documented interpretation before Track 3B"
    action         = "Do not proceed to Track 3B until the two fields are understood. Investigate the rows."

print("Interpretation:", interpretation)
print("Action        :", action)

# Guards: the EDA Report text in Section 8 assumes the verified result below.
# If the dataset changes, these fail and the text must be rewritten.
assert n_bad == 0 and n_reverse == 0 and n_price_mismatch == 0
assert n_md == 30_651 and n_total - n_md == 69_349
assert len(sal_bad) == 0, "Salami zero-discount rows found: rewrite the Salami line in Section 8"
assert round(disc_md.min(), 2) == 0.10 and round(disc_md.max(), 2) == 0.75

Interpretation: NOT A CONTRADICTION: the fields are fully consistent
Action        : No rows excluded or flagged. Use markdown_applied and discount_pct as they are.


## 8. Resolution for the EDA Report (paste-ready)

> **Data Integrity Check: `markdown_applied` vs `discount_pct` (M1-T02)**
>
> **Finding.** The v4 proposal reported rows where `markdown_applied = 1` but `discount_pct = 0` (example: a Salami batch). Checked on the downloaded file (100,000 rows), this combination occurs in **0 rows (0.00 % of all rows, 0.00 % of markdown rows)**. The reverse case, `discount_pct > 0` with `markdown_applied = 0`, also occurs in **0 rows**.
>
> **Supporting evidence.**
> - 30,651 rows have `markdown_applied = 1`; every one has a discount between **0.10 and 0.75**. The other 69,349 rows all have `discount_pct = 0`.
> - `selling_price` differs from `base_price` in 100 % of markdown rows and 0 % of non-markdown rows. The implied discount `1 − selling_price/base_price` matches `discount_pct` within cent-rounding for every row.
> - No Salami row with `markdown_applied = 1` has a zero discount.
>
> **Interpretation.** The two fields are consistent: `markdown_applied` is exactly the indicator `discount_pct > 0`. This is **not a contradiction** and does not need to be reinterpreted as "eligibility vs. redeemed". The quirk in the v4 proposal was not reproduced in the downloaded file.
>
> **Resolution.** No rows are excluded or flagged. `markdown_applied` and `discount_pct` are used as-is in Milestone 2 (discount/markdown history flags) and Track 3B. The proposal text should be corrected to say the quirk was checked and not found.
>
> **Note for Track 3B / 3C.** `discount_pct` is a fraction (0.10 = 10 %), and no batch has a discount between 0 and 0.10. The candidate-discount grid for the optimizer should be 0 plus values within [0.10, 0.75], not arbitrary values below 10 %.
>
> **Discount depth.** 60.0 % of markdown rows have a discount of 30 % or less, and 21.0 % (6,443 rows) are above 50 %. The distribution is uneven: roughly 850-900 rows per percentage point between 10 % and 30 %, a dip to about 230 per point between 30 % and 40 %, then roughly 290-350 per point from 40 % to 70 %. Support thins just below the 75 % cap (34-49 rows per step at 71-74 %) and spikes at exactly 75 % (221 rows), which looks like clipping. Overall counts are adequate up to about 70 %, but support for any particular category x expiry combination is much smaller, so Track 3B/3C must apply the support-threshold confidence flag per batch type rather than rely on these overall totals.


## 9. Save the resolution record

In [9]:
resolution = {
    "task": "M1-T02",
    "dataset_sha256": hashlib.sha256(open(DATA_PATH, "rb").read()).hexdigest(),
    "rows_total": n_total,
    "rows_markdown_applied_1": n_md,
    "rows_markdown1_and_discount0": n_bad,
    "pct_of_all_rows": round(pct_all, 4),
    "pct_of_markdown_rows": round(pct_md, 4),
    "rows_discount_without_markdown": n_reverse,
    "rows_price_mismatch": n_price_mismatch,
    "min_discount_when_markdown": float(disc_md.min()),
    "max_discount_when_markdown": float(disc_md.max()),
    "interpretation": interpretation,
    "action": action,
    "rows_excluded": 0,
    "depth_note": depth_note,
    "depth_band_rows": {str(k): int(v) for k, v in depth["rows"].items()},
}
REPORTS_M1_DIR.mkdir(parents=True, exist_ok=True)
with open(REPORTS_M1_DIR / "m1_t02_resolution.json", "w") as f:
    json.dump(resolution, f, indent=2)
print(json.dumps(resolution, indent=2))

{
  "task": "M1-T02",
  "dataset_sha256": "de94302b867c9debedfd45c431306623fdfc038f5ed8ca17736339b4460a6674",
  "rows_total": 100000,
  "rows_markdown_applied_1": 30651,
  "rows_markdown1_and_discount0": 0,
  "pct_of_all_rows": 0.0,
  "pct_of_markdown_rows": 0.0,
  "rows_discount_without_markdown": 0,
  "rows_price_mismatch": 0,
  "min_discount_when_markdown": 0.1,
  "max_discount_when_markdown": 0.75,
  "interpretation": "NOT A CONTRADICTION: the fields are fully consistent",
  "action": "No rows excluded or flagged. Use markdown_applied and discount_pct as they are.",
  "rows_excluded": 0,
  "depth_note": "60.0% of markdown rows have a discount of 30% or less; 6,443 rows (21.0%) are above 50%.",
  "depth_band_rows": {
    "10-20%": 9248,
    "20-30%": 9155,
    "30-40%": 2274,
    "40-50%": 3531,
    "50-60%": 3185,
    "60-75%": 3258
  }
}
